# Лабораторная работа 5 - Ансамбли моделей машинного обучения. Часть 1.


## 1. Загрузка данных и предварительный анализ
Загружаем датасет, проверяем пропуски.
Определяем список признаков и целевую переменную.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Загрузка данных
df = sns.load_dataset('titanic')
print("Размер датасета:", df.shape)
print("Пропуски:\n", df.isnull().sum())

# Признаки и целевая переменная
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'

# Удаляем строки, где пропущена целевая переменная (если есть)
df = df.dropna(subset=[target])
X = df[features]
y = df[target]

Размер датасета: (891, 15)
Пропуски:
 survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


##2. Обработка пропусков и кодирование категориальных признаков
Числовые признаки: заполняем пропуски медианой, затем масштабируем.
Категориальные признаки: заполняем пропуски самой частой категорией, применяем One-Hot Encoding.
Оформляем всё в ColumnTransformer, который далее будем использовать в пайплайнах моделей.

In [ ]:
# Разделяем признаки по типам
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['sex', 'embarked', 'pclass']

# Трансформер для числовых признаков
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Трансформер для категориальных признаков
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Общий препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

##3. Разделение на обучающую и тестовую выборки
Выделяем 20% данных для финальной проверки.
Стратификация гарантирует сохранение пропорции выживших/погибших в обеих выборках.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

##4. Создание и обучение ансамблевых моделей
Создаём четыре модели, каждую — в виде пайплайна: препроцессор + классификатор.
Две модели группы бэггинга: Random Forest и Extra Trees.
AdaBoost и Gradient Boosting — представители бустинга.
Для воспроизводимости фиксируем random_state.

In [ ]:
# Случайный лес (бэггинг)
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_pipe.fit(X_train, y_train)

# Сверхслучайные деревья (бэггинг)
et_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ExtraTreesClassifier(n_estimators=100, random_state=42))
])
et_pipe.fit(X_train, y_train)

# AdaBoost
ada_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', AdaBoostClassifier(n_estimators=100, random_state=42))
])
ada_pipe.fit(X_train, y_train)

# Градиентный бустинг
gb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42))
])
gb_pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'fare', 'sibsp',
                                                   'parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'embarked',
                                                   'pclass'])])),
                ('classifier', GradientBoostingClassifier(random_state=42))])

##5. Оценка качества моделей по метрике Accuracy
Для каждой модели предсказываем метки на тестовой выборке и вычисляем долю правильных ответов.
Выводим результаты для сравнения.

In [ ]:
models = {
    'Random Forest': rf_pipe,
    'Extra Trees': et_pipe,
    'AdaBoost': ada_pipe,
    'Gradient Boosting': gb_pipe
}

print("    Сравнение моделей (Accuracy на тестовой выборке)    ")
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:25} | Accuracy: {acc:.4f}")

    Сравнение моделей (Accuracy на тестовой выборке)    
Random Forest             | Accuracy: 0.8101
Extra Trees               | Accuracy: 0.7989
AdaBoost                  | Accuracy: 0.7989
Gradient Boosting         | Accuracy: 0.7933


##6. Вывод по сравнению моделей


Ожидается, что ансамблевые методы покажут близкие результаты, при этом бустинги могут немного превзойти бэггинг-модели на данном датасете. Различия в accuracy обычно составляют 1–2%, что подтверждает эффективность ансамблей на табличных данных.